Панова Света 09.07.14п2 ПР6

Импортируем библиотеки

In [63]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Загрузим датасет и выведем первые 5 строк

In [64]:
df = pd.read_csv("/content/titanic.csv")

In [65]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [66]:
df.shape

(891, 12)

Определим признаки и целевую переменную

In [67]:
features = ["Pclass", "Age", "Sex", "Fare", "SibSp", "Parch"]
X = df[features].copy()
y = df["Survived"]

Заполним пропуски в столбце возраст средним значением и заменим столбец пол на числовые значения

In [68]:
X["Age"] = X["Age"].fillna(X["Age"].median())
X["Sex"] = X["Sex"].map({"male": 0, "female": 1})

Проверяем что пропусков больше нет

In [69]:
X.isnull().sum()

,0
Pclass,0
Age,0
Sex,0
Fare,0
SibSp,0
Parch,0


Проверим что все столбцы стали числовыми

In [70]:
X.dtypes

,0
Pclass,int64
Age,float64
Sex,int64
Fare,float64
SibSp,int64
Parch,int64


Разделим на тестовую и обучающую выборки

In [71]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Выполним стандартизацию числовых признаков

In [72]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [73]:
print("Размер X_train:", X_train.shape)
print("Размер X_test:", X_test.shape)

Размер X_train: (712, 6)
Размер X_test: (179, 6)


Задание 1

По размеру выборки мы убедились что обучающая содержит примерно 80% (712), а тестовая 20% (179)

Мы оставляем тестовую выборку отдельно, чтобы проверить как модель работает на новых данных. Модель может просто запомнить какие-то данные и взаимосвязи и из-за этого завышать показатели. На тестовой выборке мы проверяем, насколько точно работает модель

scaler.fit_transform() используется только для обучающей выборки потому что fit считает параметры (среднее и std) только по обучающим данным. Если считать их по тесту — будет утечка данных, и оценка качества станет нечестной. Тест надо преобразовывать теми же параметрами через transform(), чтобы шкала была одинаковой.



Импортируем библиотеку

In [74]:
from sklearn.linear_model import LogisticRegression

Обучение логистической регрессии

In [75]:
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

Получим предсказания

In [76]:
y_pred_lr = model_lr.predict(X_test_scaled)

print("Первые предсказания:", y_pred_lr[:10])
print("Правильные ответы:", y_test.iloc[:10].to_list())

Первые предсказания: [0 0 0 0 1 0 1 0 0 0]
Правильные ответы: [0, 0, 1, 0, 1, 1, 1, 0, 0, 0]


Задание 3

0 — пассажир не выжил (погиб);
1 — пассажир выжил.


---

В первом предсказании две ошибки

Оценка качества модели

In [77]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

In [80]:
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1:", f1_score(y_test, y_pred_lr))
print("Матрица ошибок:\n", confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Accuracy: 0.8044692737430168
Precision: 0.765625
Recall: 0.7101449275362319
F1: 0.7368421052631579
Матрица ошибок:
 [[95 15]
 [20 49]]
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       110
           1       0.77      0.71      0.74        69

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



Задание 4

Доля правильных предсказаний (Accuracy) ≈ 80%.

Метрика, показывающая, насколько модель находит реально выживших, — Recall (≈ 0,71).

Ложноположительные (FP) — предсказали «выжил», а пассажир погиб.
Ложноотрицательные (FN) — предсказали «погиб», а пассажир выжил.


Задание 5

Обучение дерево решений

Загружаем библиотеку

In [81]:
from sklearn.tree import DecisionTreeClassifier

Обучаем модель

In [82]:
model_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
model_tree.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=4, random_state=42)

In [83]:
y_pred_tree = model_tree.predict(X_test)

Оцениваем модель

In [84]:
print("Accuracy дерева:", accuracy_score(y_test, y_pred_tree))
print("Precision:", precision_score(y_test, y_pred_tree))
print("Recall:", recall_score(y_test, y_pred_tree))
print("F1:", f1_score(y_test, y_pred_tree))
print("Матрица ошибок:\n", confusion_matrix(y_test, y_pred_tree))

Accuracy дерева: 0.7821229050279329
Precision: 0.875
Recall: 0.5072463768115942
F1: 0.6422018348623854
Матрица ошибок:
 [[105   5]
 [ 34  35]]


Логистическая регрессия показала более высокое качество (Accuracy 80,4%, F1 0,737) по сравнению с деревом решений (Accuracy 78,2%, F1 0,642). Дерево решений показало высокую точность (Precision 87,5%), но низкую полноту (Recall 50,7%), пропустив много выживших.

Связь с корреляционным анализом

In [85]:
corr = df[["Survived", "Pclass", "Age", "Sex", "Fare", "SibSp", "Parch"]].copy()
corr["Sex"] = corr["Sex"].map({"male": 0, "female": 1})

Выводим результаты

In [86]:
print(corr.corr(numeric_only=True)["Survived"].drop("Survived").sort_values(ascending=False))

Sex       0.543351
Fare      0.257307
Parch     0.081629
SibSp    -0.035322
Age      -0.077221
Pclass   -0.338481
Name: Survived, dtype: float64


Наиболее сильная положительная корреляция с Survived — Sex: = 0,54.

Наиболее сильная отрицательная корреляция — Pclass: = −0,34.